# tpu-retrieval-bench on Colab TPU

Runtime > Change runtime type > TPU v5e-1. One cell runs the full benchmark grid (the Run-3 protocol, 20 repetitions per record) through `scripts/reproduce_all.sh` into `results/tpu/run-<date>`, validates and summarizes it. Everything is zipped at the end.

The notebook checks out a pinned release (`REF`) and installs the locked dependency versions (`notebooks/constraints.txt`, exported from `uv.lock`), so a rerun uses the code and libraries of that release.

In [ ]:
import sys
print(sys.version)
if sys.version_info < (3, 12):
    print('WARNING: python < 3.12. Fallback: pip install "jax[tpu]<0.11" and '
          'pip install --ignore-requires-python -e ".[data,plot]" (repo declares >= 3.12).')

In [ ]:
# Pure Python, no jax import (the notebook kernel must never hold the TPU).
import datetime

# Release tag to reproduce. Bump this when a newer release is cut (or pass a commit SHA).
REF = "v0.2.0"
# Output directory: the canonical results location. Override to rerun into another directory.
RUN_ID = "results/tpu/run-" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M")
print(REF, RUN_ID)

In [ ]:
!git clone https://github.com/maxpetrusenko/tpu-retrieval-bench
%cd tpu-retrieval-bench
!git checkout --detach {REF}
!pip install -q -c notebooks/constraints.txt -e ".[data,plot]" "jax[tpu]==0.12.0.1" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
# Dependencies are held to uv.lock through the constraints file (regenerate it with
# scripts/export_constraints.sh); jax[tpu] is pinned to the locked jax version (0.12.0.1).
# If pip changed jax, restart the runtime (Runtime > Restart session) before running the next cell.

In [ ]:
# Run in a subprocess: importing jax in the notebook kernel would lock the TPU and make every
# later `!python` cell fail with "TPU is already in use".
!python -c "import jax; d = jax.devices(); assert d[0].platform == 'tpu', f'not a TPU runtime: {d}'; print(d[0].device_kind, len(d), 'device(s)')"

## Full benchmark grid (single entry point)

Main grid, SIFT-1M, block sweeps, recall sweep and device time, written to `RUN_ID` and `RUN_ID-devtime`. Takes about 30 minutes; the console can stay quiet for long stretches while a step runs. `scripts/reproduce_all.sh` is also the local command.

In [ ]:
# PYTHON=python3: use this runtime's jax[tpu], not a separate uv environment.
!PYTHON=python3 bash scripts/reproduce_all.sh {RUN_ID}

## Profiler trace for one config

In [ ]:
# Script, not an in-kernel cell: the kernel must never import jax (it would hold the TPU).
!python bench/profile_one.py --m 128 --n 1000000 --d 128 --k 10 --out results/profile_jit_topk_m128_n1e6_d128

## Plots and download

In [ ]:
# Plots go next to the run, in an untracked directory: writing to docs/img would overwrite
# tracked images and leave the checkout dirty. Maintainers refresh docs/img with
# scripts/regen_docs_img.sh.
!python bench/plot.py --results {RUN_ID} --out {RUN_ID}/plots

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")